# Transfer: across samples (mouse brain) and across technologies (PBMC)

`transfer` annotates a query from a labelled reference in one call. It is made for a reference and a query that
differ in sample, depth or technology. This notebook runs two transfers, each followed by its report:

1. **Across samples, mouse brain:** the Allen Mouse Brain cortex data, two regions profiled with the same
   technology (Smart-seq v4): the primary visual cortex (**VISp**) and the anterior lateral motor cortex (**ALM**),
   annotated at 34 cell types (AMB34). VISp → ALM, then ALM → VISp.
2. **Across technologies, PBMC:** one donor's blood cells (pbmc1) profiled with **Smart-seq2** (253 cells, deep) and
   **10x Chromium v3** (shallow): Smart-seq2 → 10x v3, on the cell types both share.

A last section puts back the 10x v3 cells whose population the Smart-seq2 reference **lacks** and shows what happens
to them.

**Why these pairs.** Both come from SCARLET's formal benchmark, which reruns the benchmark of Abdelaal et al.
(*Genome Biology* 20, 194, 2019): AMB34 is its brain task (benchmark task `B/Brain_MouseV1_MouseALM_Labels34`),
matched labels, the same 34 types in both regions; Smart-seq2 → 10x v3 is one of its PBMC pairs (task
`B/PbmcBench_Smart-Seq2`), with labels matched across two genuinely different technologies. All pairs are in
[BENCHMARKS](../docs/benchmarks/index.md). The benchmark ran without the blocklist (all tools saw the same genes);
this notebook keeps SCARLET's defaults, so its numbers can differ slightly from the benchmark's.

**Time and size:** the brain data are large (12,552 and 8,128 cells, about 11,000 detected genes per cell; 677 MB
to download). On a laptop with 4 CPUs each brain transfer takes a few minutes and its report a few more; the PBMC
part takes about a minute. Guides: [transfer](../docs/guide/transfer.md), [reports](../docs/reports/index.md),
[reading the output](../docs/output.md#transfer).

## Setup

`NCPU` sets how many CPU cores SCARLET may use: a transfer runs that many of its draws side by side, each with its
own copy of the data, so more CPUs are faster but take more memory; it never changes a result. This notebook uses
**2**, which fits a laptop. Measured with 2 CPUs, SCARLET alone (Jupyter and the browser come on top): this
notebook's own process peaks at about 5 GB; with the two processes that run a transfer's draws beside it, SCARLET's
processes together peak at about 7.5 GB (memory they share counted once). With 4 CPUs it did not fit on a laptop
beside other programs. Raise it only if you have that memory free.

Run one notebook at a time: a notebook keeps its memory until it is shut down; stopping its cells or closing its tab
does not free it. Close the others first (File → Close and Shut Down Notebook).

In [ ]:
import os

import scarlet

print("SCARLET", scarlet.__version__)
NCPU = min(2, os.cpu_count() or 1)   # with 2: about 5 GB in this process, about 7.5 GB with the draws' processes
os.makedirs("out", exist_ok=True)

## The data

`scarlet.datasets.fetch` downloads each file once and checks its SHA-256. The files are the benchmark's inputs with
the benchmark's own cell selection, compressed (their content is checked to be the benchmark's: `MANIFEST.tsv` in
the release assets): `amb34_visp.h5ad` and `amb34_alm.h5ad` (task `B/Brain_MouseV1_MouseALM_Labels34`, labels in
`obs['cell_type_level34']`); `pbmc1_smartseq2.h5ad` and `pbmc1_10xv3_query.h5ad` (task `B/PbmcBench_Smart-Seq2`:
the Smart-seq2 reference and the benchmark's 10x v3 query, the 2,990 cells whose population Smart-seq2 has);
`pbmc1_10xv3.h5ad` (task `B/PbmcBench_10Xv3`: the whole 10x v3 sample, 3,222 cells, for the last section).

The query files keep their labels in `obs`. `transfer` reads only the counts of a query (and an origin column, when
you name one), never its labels; the reports use them only in the section marked *labels supplied by the user*,
to score the calls.

In [ ]:
from scarlet import datasets

paths = datasets.fetch(["amb34_visp.h5ad", "amb34_alm.h5ad", "pbmc1_smartseq2.h5ad", "pbmc1_10xv3_query.h5ad",
                        "pbmc1_10xv3.h5ad"], dest="data")

## What transfer does

1. **Depth matching** (on by default): the deeper side is thinned towards the shallower one, so markers are learned
   and scored at comparable depth.
2. **Technology preset**: chosen from the shallower side (dense at 5,000 detected genes per cell or more, else
   sparse). Both are printed in the pre-flight summary.
3. **First pass:** learn on the reference, assign the query.
4. **Refinement** (on by default): the *stable anchors* of a cell type are the query cells the first pass called
   uniquely to it in every draw. A cell type with at least 3 stable anchors is learned again from them, on the query
   itself, and the query is assigned again. Cell types with fewer anchors are **not refined** and keep their
   first-pass calls.
5. **Draws:** steps 1, 3 and 4 run 10 times with different seeds (accurate mode, the default). Each cell's
   **agreement** is how many of the 10 draws gave its most frequent outcome (`unassigned` counts as an outcome). The
   final call needs at least **8 of 10**; below that the cell is `unassigned`, reason *undetermined across draws*.
   Agreement measures how stable a call is under the random thinning and seeds of the draws, **not** whether it is
   correct. `draws=1` is quick mode: one draw, much faster, with seed-dependent calls.

At the default verbosity SCARLET prints the pre-flight once, one progress bar, and one line per draw.

## 1. Across samples: VISp → ALM

The reference is VISp with its labels (`cell_type_level34`); the query is ALM. Command-line twin:
```
scarlet transfer --reference data/amb34_visp.h5ad --labels cell_type_level34 --query data/amb34_alm.h5ad \
                 --output out/visp_to_alm
```

In [ ]:
res = scarlet.transfer(paths["amb34_visp.h5ad"], "cell_type_level34", paths["amb34_alm.h5ad"],
                       "out/visp_to_alm", ncpu=NCPU)

The report of this transfer. `query_labels` names the query's own labels, which the report compares the calls with
in a section marked as labels supplied by you (SCARLET did not use them). The report also writes a supplement, the
learn report of the reference model, whose cross-validation learns and assigns five times on VISp: that is most of
the report's time.

Command-line twin:
```
scarlet report out/visp_to_alm --query-labels cell_type_level34
```

In [ ]:
html = scarlet.report("out/visp_to_alm", query_labels="cell_type_level34", ncpu=NCPU)
print("report:", html)

The scores, read together as in the learn/assign notebooks: the share refused, and the median F1 per cell type on
the called cells (refusals are not errors). *final* is after refinement, *first pass* before it.

In [ ]:
import pandas as pd


def scores(report_html):
    """The report's scores against the labels supplied by the user (its bundle table known_summary.tsv)."""
    tables = os.path.join(os.path.dirname(report_html), "bundle", "tables")
    return pd.read_csv(os.path.join(tables, "known_summary.tsv"), sep="\t")[
        ["output", "cells", "pct_refused", "median_f1_called"]].round(3)


display(scores(html))

### Agreement across draws, on a few cells

`labels.tsv` holds the final calls with their agreement. Below, a few cells left unassigned because no outcome won
at least 8 of the 10 draws (*undetermined across draws*), next to cells called at full agreement. Their agreement
says how often the most frequent outcome came up; the report's section 7 shows the whole distribution.

In [ ]:
labels = pd.read_csv("out/visp_to_alm/labels.tsv", sep="\t", index_col=0)
cols = ["scarlet_best", "scarlet_best_agreement", "scarlet_best_reason", "scarlet_top1", "scarlet_runner_up"]
undetermined = labels[labels.scarlet_best_reason == "undetermined across draws"]
display(undetermined[cols].head(3))
display(labels[labels.scarlet_best_agreement == 10][cols].head(2))

### The other direction: ALM → VISp

The same transfer with the regions swapped. Its report skips the self-assessment (`self_assessment=False`) to save
time; the scores against the labels are the same kind.

In [ ]:
res_back = scarlet.transfer(paths["amb34_alm.h5ad"], "cell_type_level34", paths["amb34_visp.h5ad"],
                            "out/alm_to_visp", ncpu=NCPU)
html_back = scarlet.report("out/alm_to_visp", query_labels="cell_type_level34", self_assessment=False, ncpu=NCPU)
display(scores(html_back))

## 2. Across technologies: Smart-seq2 → 10x v3

A small, deep reference (253 Smart-seq2 cells, about 2,400 detected genes per cell) and a shallow query (10x v3,
about 1,500). The query holds only the cell types the reference has, as in the benchmark: the benchmark's rule
(Abdelaal et al.) drops the query cells whose label the reference lacks, here natural killer and dendritic cells.

Command-line twins:
```
scarlet transfer --reference data/pbmc1_smartseq2.h5ad --labels cell_type --query data/pbmc1_10xv3_query.h5ad \
                 --output out/smartseq2_to_10xv3
scarlet report out/smartseq2_to_10xv3 --query-labels cell_type
```

In [ ]:
res_pbmc = scarlet.transfer(paths["pbmc1_smartseq2.h5ad"], "cell_type", paths["pbmc1_10xv3_query.h5ad"],
                            "out/smartseq2_to_10xv3", ncpu=NCPU)

In [ ]:
html_pbmc = scarlet.report("out/smartseq2_to_10xv3", query_labels="cell_type", ncpu=NCPU)
print("report:", html_pbmc)
display(scores(html_pbmc))

The scores per cell type (the report's bundle table `known_per_label.tsv`): the share called correctly, called
as another cell type, and refused, for the final best and unique calls.

In [ ]:
per_label = pd.read_csv(os.path.join(os.path.dirname(html_pbmc), "bundle", "tables", "known_per_label.tsv"),
                        sep="\t")
display(per_label[["label", "cells", "pct_correct, final best", "pct_refused, final best",
                   "pct_correct, final unique", "pct_refused, final unique"]].round(1))

## 3. When the reference lacks a population

Now the whole 10x v3 sample, with the cells the benchmark drops: 194 natural killer (NK) cells and 38 dendritic
cells, two populations the Smart-seq2 reference does not have. (This 10x v3 sample has no plasmacytoid dendritic
cells.) The same transfer, then the report against the query's labels.

In [ ]:
res_all = scarlet.transfer(paths["pbmc1_smartseq2.h5ad"], "cell_type", paths["pbmc1_10xv3.h5ad"],
                           "out/smartseq2_to_10xv3_all", ncpu=NCPU)
html_all = scarlet.report("out/smartseq2_to_10xv3_all", query_labels="cell_type", self_assessment=False, ncpu=NCPU)
per_label_all = pd.read_csv(os.path.join(os.path.dirname(html_all), "bundle", "tables", "known_per_label.tsv"),
                            sep="\t")
display(per_label_all[["label", "known_to_model", "cells", "most frequent call, final best",
                       "pct_other identity, final best", "pct_refused, final best"]].round(1))

**What this shows.** A cell of a population the reference lacks has no correct label: the right outcome is a
refusal. SCARLET refuses a cell when the background explains it as well as any cell type of the reference. A
population *close* to one the reference has fits that relative better than the background: here almost every NK cell
is called a cytotoxic T cell (the two share many cytotoxic genes). The dendritic cells, further from every cell type
of this reference, are split: about half are refused, the rest called as another cell type. A population *distant*
from everything in the reference is refused for background: that is what the benchmark's rejection controls test
(Fig. 6 of Abdelaal et al.: a reference of one tissue and a query of another), where SCARLET refuses nearly every
cell.

**For your data:** a reference-based method, SCARLET included, can only call the cell types its reference has. A
population close to one of them is called as its relative; check the cells of the cell types you expect a missing
relative for (here the cytotoxic T cells), for example in the report's UMAP on all genes (its *space* switch),
where such a population may stand apart, or use a reference that has it. This is a limit of reference-based
annotation itself, not of one tool (the benchmark's own test on populations missing from the reference is in
[BENCHMARKS](../docs/benchmarks/index.md)).

## Export

Each transfer folder holds `labels.tsv` (the final calls, one row per query cell, with agreement, reasons, top
identity and runner-up), `first_pass.tsv` (before refinement), `qc.tsv` (the transfer check), the models,
`summary.txt` and `transfer_record.json` (the full record, with the inputs' SHA-256); the report folder holds
`report.html` and its bundle of tables.

In [ ]:
for folder in ("out/visp_to_alm", "out/alm_to_visp", "out/smartseq2_to_10xv3", "out/smartseq2_to_10xv3_all"):
    print(os.path.abspath(f"{folder}/labels.tsv"))
display(pd.read_csv("out/smartseq2_to_10xv3/labels.tsv", sep="\t", index_col=0).iloc[:5, :8])